# Classification on imbalanced data

Part of the code was taken from the [tensorflow tutorial](https://www.tensorflow.org/tutorials/structured_data/imbalanced_data).
We use [Keras](../../guide/keras/overview.ipynb) to define the model and [class weights](https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/Model) to help the model learn from the imbalanced data.

This demonstrates how to classify a highly imbalanced dataset in which the number of examples in one class greatly outnumbers the examples in another. The aim is to detect a mere 10001 attack packets from 298,278 packets in total.

## Setup

In [ ]:
#%%writefile anomaly-detector.py
import tensorflow as tf
from tensorflow import keras
from datetime import datetime
from packaging import version

import os
import tempfile

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

import sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, plot_confusion_matrix, plot_precision_recall_curve
from sklearn.utils.multiclass import unique_labels
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.wrappers.scikit_learn import KerasClassifier

from sklearn.model_selection import cross_val_score, cross_validate, cross_val_predict
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
import mlflow
import mlflow.keras
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

import time
mlflow.keras.autolog()

#np.random.seed(5)

In [ ]:
#%%writefile -a anomaly-detector.py

mpl.rcParams['figure.figsize'] = (12, 10)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

In [ ]:
#%%writefile -a anomaly-detector.py

def print_table(title, scores_orig, scores_under, scores_over):
    metrics = {
        'F1': 'test_f1',
        'ROC-AUC': 'test_roc_auc',
        'Average Precision': 'test_average_precision',
        'Balanced Accuracy': 'test_balanced_accuracy',
        'Precision': 'test_precision',
        'Recall': 'test_recall',
        'Fit Time': 'fit_time',
    }

    print("== %s: == " % (title))
    print()
    print("Metric,Original Mean,Original Std,Undersampled Mean,Undersampled Std,Oversampled Mean,Oversampled Std")
    
    for key, value in metrics.items():
        print("%s,%.5f,%.5f,%.5f,%.5f,%.5f,%.5f" % (key, scores_orig[value].mean(), scores_orig[value].std(), scores_under[value].mean(), scores_under[value].std(), scores_over[value].mean(), scores_over[value].std()))

## Data processing and exploration

In [ ]:
#%%writefile -a anomaly-detector.py

# Import training data
dataset_train = pd.read_csv("../data/raw/energy-informatics-2020/csvDataFeaturesTrain.csv", delimiter=';')
dataset_test = pd.read_csv("../data/raw/energy-informatics-2020/csvDataFeaturesTest.csv", delimiter=';')

#data_train = dataset_test.append(dataset_test[dataset_test['attack']==1])
X = dataset_train.append(dataset_test)
X.groupby('attack').count()

# Create Training Dataset
- attack = 0 means normal
- attack = 1 means anomaly

In [ ]:
#%%writefile -a anomaly-detector.py

X['attack'] = X['attack'].map({0:0, 1:1, 2:1, 3:1, 4:1})
print(np.bincount(X['attack']))

### Examine the class label imbalance

Let's look at the dataset imbalance:

In [ ]:
#%%writefile -a anomaly-detector.py

neg, pos = np.bincount(X['attack'])
total = neg + pos
print('Examples:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

This shows the small fraction of positive samples.

In [ ]:
#%%writefile -a anomaly-detector.py

# Use a utility from sklearn to split and shuffle our dataset.
y = X.pop('attack')
X

X_under, y_under = RandomUnderSampler(sampling_strategy='majority').fit_resample(X, y)
X_over, y_over = RandomOverSampler().fit_resample(X, y)

print(X.shape)
print(X_under.shape)
print(np.bincount(y_under))
print(X_over.shape)
print(np.bincount(y_over))

In [ ]:
#%%writefile -a anomaly-detector.py

def print_balance(y, title):
    classes = np.unique(y)
    total = len(y)
    print(title)
    for c in classes:
        n_examples = len(y[y==c])
        percent = n_examples / total * 100
        print('  > Class=%d : %d/%d (%.1f%%)' % (c, n_examples, total, percent))

print_balance(y, "Original Dataset")
print_balance(y_under, "Undersampled Dataset")
print_balance(y_over, "Oversampled Dataset")

## Define the model and metrics

Define a function that creates a simple neural network with a densly connected hidden layer, a [dropout](https://developers.google.com/machine-learning/glossary/#dropout_regularization) layer to reduce overfitting, and an output sigmoid layer that returns the probability of a transaction being fraudulent: 

In [ ]:
#%%writefile -a anomaly-detector.py

# Create a TensorBoard callback
logs = "./logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")

tboard_callback = tf.keras.callbacks.TensorBoard(log_dir = logs,
                                                 histogram_freq = 1,
                                                 profile_batch = '500,520')

METRICS = [
      keras.metrics.TruePositives(name='tp'),
      keras.metrics.FalsePositives(name='fp'),
      keras.metrics.TrueNegatives(name='tn'),
      keras.metrics.FalseNegatives(name='fn'), 
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.Precision(name='precision'),
      keras.metrics.Recall(name='recall'),
      keras.metrics.AUC(name='auc'),
      keras.metrics.AUC(name='prc', curve='PR'), # precision-recall curve,      
]

def make_model(metrics=METRICS):   
    model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(21,)),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=keras.losses.BinaryCrossentropy(), metrics=metrics)
    return model

In [ ]:
print(make_model().summary())

### Understanding useful metrics

Notice that there are a few metrics defined above that can be computed by the model that will be helpful when evaluating the performance.



*   **False** negatives and **false** positives are samples that were **incorrectly** classified
*   **True** negatives and **true** positives are samples that were **correctly** classified
*   **Accuracy** is the percentage of examples correctly classified
>   $\frac{\text{true samples}}{\text{total samples}}$
*   **Precision** is the percentage of **predicted** positives that were correctly classified
>   $\frac{\text{true positives}}{\text{true positives + false positives}}$
*   **Recall** is the percentage of **actual** positives that were correctly classified
>   $\frac{\text{true positives}}{\text{true positives + false negatives}}$
*   **AUC** refers to the Area Under the Curve of a Receiver Operating Characteristic curve (ROC-AUC). This metric is equal to the probability that a classifier will rank a random positive sample higher than a random negative sample.

Note: Accuracy is not a helpful metric for this task. You can 99.8%+ accuracy on this task by predicting False all the time.  

Read more:
*  [True vs. False and Positive vs. Negative](https://developers.google.com/machine-learning/crash-course/classification/true-false-positive-negative)
*  [Accuracy](https://developers.google.com/machine-learning/crash-course/classification/accuracy)
*   [Precision and Recall](https://developers.google.com/machine-learning/crash-course/classification/precision-and-recall)
*   [ROC-AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)

## Baseline model

### Build the model

In [ ]:
#%%writefile -a anomaly-detector.py

EPOCHS = 500
BATCH_SIZE = 4096

### Train the model

In [ ]:
#%%writefile -a anomaly-detector.py

# Source: https://scikit-learn.org/0.21/auto_examples/model_selection/plot_confusion_matrix.html

def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    # Only use the labels that appear in the data
    classes = classes[unique_labels(y_true, y_pred)]
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    # We want to show all ticks...
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           # ... and label them with the respective list entries
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax

np.set_printoptions(precision=2)

In [ ]:
#%%writefile -a anomaly-detector.py

def cross_validate_model(pipeline, X, y, cv, title):
    print_balance(y, title)
    scores = cross_validate(pipeline, X.values, y.values, cv=kfold, scoring=('f1', 'roc_auc', 'average_precision', 'balanced_accuracy', 'precision', 'recall'), return_estimator=True, return_train_score=True)
    print("    > F1: %.3f%% (+/-%.3f%%)" % (scores['test_f1'].mean()*100, scores['test_f1'].std()*100))        
    print("    > ROC-AUC: %.3f%% (+/-%.3f%%)" % (scores['test_roc_auc'].mean()*100, scores['test_roc_auc'].std()*100))
    print("    > Average Precision: %.3f%% (+/-%.3f%%)" % (scores['test_average_precision'].mean()*100, scores['test_average_precision'].std()*100))
    print("    > Balanced Accuracy: %.3f%% (+/-%.3f%%)" % (scores['test_balanced_accuracy'].mean()*100, scores['test_balanced_accuracy'].std()*100))
    print("    > Precision: %.3f%% (+/-%.3f%%)" % (scores['test_precision'].mean()*100, scores['test_precision'].std()*100))
    print("    > Recall: %.3f%% (+/-%.3f%%)" % (scores['test_recall'].mean()*100, scores['test_recall'].std()*100))
    print("    > Fit Time: %.3f s (+/-%.3f s)" % (scores['fit_time'].mean(), scores['fit_time'].std()))
    return scores

In [ ]:
#%%writefile -a anomaly-detector.py

# Create stratified KFold Splits
kfold = StratifiedKFold(n_splits=10, shuffle=True)

# Random Forest

In [ ]:
#%%writefile -a anomaly-detector.py

from sklearn.ensemble import RandomForestClassifier

rf_original = cross_validate_model(make_pipeline(StandardScaler(), RandomForestClassifier()), X, y, kfold, "Original Dataset")
rf_under = cross_validate_model(make_pipeline(StandardScaler(), RandomForestClassifier()), X_under, y_under, kfold, "Undersampled Dataset")
rf_over = cross_validate_model(make_pipeline(StandardScaler(), RandomForestClassifier()), X_over, y_over, kfold, "Oversampled Dataset")

print_table("Random Forest Classifier", rf_original, rf_under, rf_over)

In [ ]:
print_table("Random Forest Classifier", rf_original, rf_under, rf_over)

# Decision Tree

In [ ]:
#%%writefile -a anomaly-detector.py

from sklearn import tree

dt_orig = cross_validate_model(make_pipeline(StandardScaler(), tree.DecisionTreeClassifier()), X, y, kfold, "Original Dataset")
dt_under = cross_validate_model(make_pipeline(StandardScaler(), tree.DecisionTreeClassifier()), X_under, y_under, kfold, "Undersampled Dataset")
dt_over = cross_validate_model(make_pipeline(StandardScaler(), tree.DecisionTreeClassifier()), X_over, y_over, kfold, "Oversampled Dataset")

print_table("Decision Tree", dt_orig, dt_under, dt_over)

In [ ]:
print_table("Decision Tree", dt_orig, dt_under, dt_over)

# DNN Model (5 Hidden Layers, 1 Output Layer)

In [ ]:
#%%writefile -a anomaly-detector.py

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

print("Epochs: %s, Batch Size: %s" % (EPOCHS, BATCH_SIZE))
dnn_original = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)), X, y, kfold, "Original Dataset")
dnn_under = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)), X_under, y_under, kfold, "Undersampled Dataset")
dnn_over = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)), X_over, y_over, kfold, "Oversampled Dataset")

print_table("DNN", dnn_original, dnn_under, dnn_over)

In [ ]:
print_table("DNN", dnn_original, dnn_under, dnn_over)

# DNN Model with adjusted weights

In [ ]:
#%%writefile -a anomaly-detector.py

# Scaling by total/2 helps keep the loss to a similar magnitude.
# The sum of the weights of all examples stays the same.
weight_for_0 = (1 / neg)*(total)/2.0 
weight_for_1 = (1 / pos)*(total)/2.0

class_weight = {0: weight_for_0, 1: weight_for_1}

print('Weight for class 0 (normal): {:.2f}'.format(weight_for_0))
print('Weight for class 1 (attack): {:.2f}'.format(weight_for_1))

#estimators = []
#model = KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, class_weight=class_weight)
#model._estimator_type = "classifier"
#estimators.append(('standardize', StandardScaler()))
#estimators.append(('weighted-model', model))
#pipeline = Pipeline(estimators)

weighted_orig = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, class_weight=class_weight)), X, y, kfold, "Original Dataset")
weighted_under = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, class_weight=class_weight)), X_under, y_under, kfold, "Undersampled Dataset")
weighted_over = cross_validate_model(make_pipeline(StandardScaler(), KerasClassifier(build_fn=make_model, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, class_weight=class_weight)), X_over, y_over, kfold, "Oversampled Dataset")

print_table("Weighted DNN Model", weighted_orig, weighted_under, weighted_over)

In [ ]:
print_table("Weighted DNN Model", weighted_orig, weighted_under, weighted_over)